# Masked Face Verification: Pair-Head Final Notebook

This notebook packages the final project direction: a frozen FaceNet recognizer with a lightweight pair-level verifier head, compared against dedicated mask-aware models as a ceiling.

Default mode reads the committed result artifacts. Set `RUN_EXPENSIVE = True` to rerun the probes on a GPU runtime.

## Setup

Use a GPU runtime. The scripts expect the RMFRD directory to contain masked and unmasked identity folders. The previously used Colab path was:

`/content/datasets/rmfrd/extracted/self-built-masked-face-recognition-dataset`

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/tungd/masked-face-id.git"
REPO_DIR = Path("/content/masked-face-id")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/masked-face-id
!git pull --ff-only origin main
!python scripts/install_colab_deps.py

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/datasets/rmfrd/extracted/self-built-masked-face-recognition-dataset")
OUT_ROOT = Path("/content/masked_face_final_runs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Leave this False for a fast report/visualization notebook that uses committed artifacts.
# Set True only when the RMFRD data is present and the runtime has a GPU.
RUN_EXPENSIVE = False

SEEDS = [42, 7]
print({"data_root_exists": DATA_ROOT.exists(), "out_root": str(OUT_ROOT), "run_expensive": RUN_EXPENSIVE})

## Final Method

1. Align each face with MTCNN.
2. Extract frozen FaceNet embeddings for five views: full face, lower blackout, lower blur, upper-only, and eye-band.
3. Build pair features from view scores, score statistics, absolute embedding differences, and elementwise products.
4. Train a small MLP verifier head on training-identity pairs.
5. At inference, use the pair head for masked cases and bypass unmasked-unmasked pairs to the original FaceNet score.

In [ ]:
def run(cmd):
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)

if RUN_EXPENSIVE:
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")
    pair_configs = {
        42: {"epochs": 40, "dropout": 0.25, "train_pairs": 3000},
        7: {"epochs": 50, "dropout": 0.30, "train_pairs": 4000},
    }
    for seed in SEEDS:
        cfg = pair_configs[seed]
        run([
            sys.executable, "scripts/probe_pair_verifier_head.py",
            "--data-root", DATA_ROOT,
            "--out-dir", OUT_ROOT / f"pair_head_seed{seed}",
            "--train-identities", 140,
            "--eval-identities", 80,
            "--max-images-per-condition", 8,
            "--train-pairs-per-case", cfg["train_pairs"],
            "--eval-pairs-per-case", 800,
            "--epochs", cfg["epochs"],
            "--dropout", cfg["dropout"],
            "--seed", seed,
        ])
else:
    print("Skipping expensive pair-head runs; using committed artifacts below.")

## Dedicated Mask-Aware Ceiling

The dedicated comparison uses official checkpoints from `fdbtrs/Masked-Face-Recognition-KD` with 112x112 ArcFace-style alignment. This is a ceiling model: it is allowed to be purpose-built for masked recognition.

In [ ]:
if RUN_EXPENSIVE:
    for seed in SEEDS:
        run([
            sys.executable, "scripts/probe_maskaware_baseline.py",
            "--data-root", DATA_ROOT,
            "--out-dir", OUT_ROOT / f"maskaware_seed{seed}",
            "--download-missing",
            "--seed", seed,
        ])
else:
    print("Skipping expensive dedicated-model runs; using committed artifacts below.")

In [ ]:
import pandas as pd
import numpy as np

ARTIFACTS = Path("artifacts")

PAIR_ARTIFACTS = {
    42: ARTIFACTS / "rmfrd_pair_verifier_head_probe" / "pair_verifier_head_results.csv",
    7: ARTIFACTS / "rmfrd_pair_verifier_head_seed7" / "pair_verifier_head_results.csv",
}
DEDICATED_ARTIFACTS = {
    42: ARTIFACTS / "rmfrd_maskaware_baseline_seed42" / "maskaware_baseline_results.csv",
    7: ARTIFACTS / "rmfrd_maskaware_baseline_seed7" / "maskaware_baseline_results.csv",
}

def auc(df, model, case):
    row = df[(df["model"] == model) & (df["case"] == case)]
    return float(row.iloc[0]["roc_auc"])

rows = []
for seed in SEEDS:
    pair = pd.read_csv(PAIR_ARTIFACTS[seed])
    dedicated = pd.read_csv(DEDICATED_ARTIFACTS[seed])
    baseline_mu = auc(pair, "baseline_full", "masked-unmasked")
    pair_mu = auc(pair, "pair_head_masked_cases_only", "masked-unmasked")
    baseline_uu = auc(pair, "baseline_full", "unmasked-unmasked")
    pair_uu = auc(pair, "pair_head_masked_cases_only", "unmasked-unmasked")
    dedicated_mu = dedicated[dedicated["case"] == "masked-unmasked"]
    dedicated_mu = dedicated_mu[dedicated_mu["model"] != "baseline_facenet_full"].sort_values("roc_auc", ascending=False).iloc[0]
    dedicated_uu = auc(dedicated, dedicated_mu["model"], "unmasked-unmasked")
    rows.extend([
        {"seed": seed, "model": "FaceNet baseline", "masked_unmasked_auc": baseline_mu, "gain": np.nan, "unmasked_unmasked_auc": baseline_uu},
        {"seed": seed, "model": "Pair head masked-only", "masked_unmasked_auc": pair_mu, "gain": pair_mu - baseline_mu, "unmasked_unmasked_auc": pair_uu},
        {"seed": seed, "model": f"Dedicated ceiling ({dedicated_mu['model']})", "masked_unmasked_auc": float(dedicated_mu["roc_auc"]), "gain": float(dedicated_mu["roc_auc"]) - baseline_mu, "unmasked_unmasked_auc": dedicated_uu},
    ])

summary = pd.DataFrame(rows)
summary

In [ ]:
import matplotlib.pyplot as plt

plot_df = summary.copy()
plot_df["label"] = plot_df["seed"].astype(str) + " / " + plot_df["model"]

fig, ax = plt.subplots(figsize=(10, 4.8))
colors = plot_df["model"].map(lambda name: "#596675" if "FaceNet" in name else ("#2364aa" if "Pair" in name else "#26845b"))
ax.barh(plot_df["label"], plot_df["masked_unmasked_auc"], color=colors)
ax.set_xlim(0.70, 0.90)
ax.set_xlabel("Masked-unmasked ROC-AUC")
ax.set_title("Pair-head adapter improves FaceNet, dedicated model remains the ceiling")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
for seed in SEEDS:
    face = summary[(summary.seed == seed) & (summary.model == "FaceNet baseline")].iloc[0]
    pair = summary[(summary.seed == seed) & (summary.model == "Pair head masked-only")].iloc[0]
    ceiling = summary[(summary["seed"] == seed) & (summary["model"].str.startswith("Dedicated ceiling"))].iloc[0]
    print(f"Seed {seed}: pair head improves masked-unmasked ROC-AUC by {pair.gain:+.4f}; dedicated ceiling remains {ceiling.masked_unmasked_auc - pair.masked_unmasked_auc:+.4f} above it.")

print("\nFinal framing: lightweight adaptation for existing unmasked recognizers, not a replacement for purpose-built mask-aware recognizers.")

## Deliverables

- Report: `docs/final-report.md`
- Slides: `slides/pair_head_final_presentation.html`
- Primary probe: `scripts/probe_pair_verifier_head.py`
- Dedicated ceiling probe: `scripts/probe_maskaware_baseline.py`
- Saved result artifacts: `artifacts/rmfrd_pair_verifier_head_*` and `artifacts/rmfrd_maskaware_baseline_*`